<a href="https://colab.research.google.com/github/AshokGit544/Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline/blob/main/Airflow_PySpark_SAP_FICO_Common_Key_Retrieval_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install pandas numpy

import os
import random
import zipfile
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

random.seed(42)
np.random.seed(42)

PROJECT_DIR = Path("Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline")
RAW_DIR = PROJECT_DIR / "data" / "raw"
OUT_DIR = PROJECT_DIR / "data" / "output"
DAG_DIR = PROJECT_DIR / "dags"

RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
DAG_DIR.mkdir(parents=True, exist_ok=True)


# Create synthetic SAP FICO-style master data

vendors = pd.DataFrame([
    ["V001", "Alpha Energy Services", "Maintenance", "NY"],
    ["V002", "BlueGrid Technologies", "IT Services", "CT"],
    ["V003", "Northwind Electric", "Electrical Components", "MA"],
    ["V004", "GreenVolt Contractors", "Field Services", "PA"],
    ["V005", "Prime Utility Logistics", "Logistics", "NJ"]
], columns=["vendor_id", "vendor_name", "vendor_category", "vendor_state"])

gl_accounts = pd.DataFrame([
    ["400100", "Revenue - Energy Delivery", "Revenue"],
    ["500100", "Opex - Maintenance", "Expense"],
    ["500200", "Opex - IT Services", "Expense"],
    ["500300", "Opex - Contractors", "Expense"],
    ["500400", "Opex - Logistics", "Expense"],
    ["110100", "Accounts Payable", "Liability"]
], columns=["gl_account", "gl_account_name", "gl_type"])

cost_centers = pd.DataFrame([
    ["CC100", "Transmission Operations", "Operations"],
    ["CC110", "Distribution Maintenance", "Operations"],
    ["CC120", "Substation Engineering", "Engineering"],
    ["CC130", "Customer Support", "Customer Ops"],
    ["CC140", "Finance Operations", "Finance"]
], columns=["cost_center_id", "cost_center_name", "department"])

vendors.to_csv(RAW_DIR / "vendors.csv", index=False)
gl_accounts.to_csv(RAW_DIR / "gl_accounts.csv", index=False)
cost_centers.to_csv(RAW_DIR / "cost_centers.csv", index=False)


# Create invoice and payment transactions

invoice_rows = []
payment_rows = []

start_date = datetime(2024, 1, 1)

descriptions = [
    "transformer repair service",
    "sap fico support work",
    "field contractor invoice",
    "substation maintenance work",
    "it platform support",
    "logistics support for equipment",
    "distribution network inspection"
]

for i in range(1, 101):
    vendor = vendors.sample(1).iloc[0]
    gl = gl_accounts[gl_accounts["gl_type"].isin(["Expense", "Revenue"])].sample(1).iloc[0]
    cc = cost_centers.sample(1).iloc[0]

    posting_date = start_date + timedelta(days=random.randint(0, 365))
    invoice_date = posting_date - timedelta(days=random.randint(1, 10))
    amount = round(random.uniform(1000, 50000), 2)

    source_doc_id = f"SRC{i:04d}"
    invoice_id = f"INV{i:05d}"

    invoice_rows.append([
        invoice_id,
        source_doc_id,
        vendor["vendor_id"],
        gl["gl_account"],
        cc["cost_center_id"],
        invoice_date.date().isoformat(),
        posting_date.date().isoformat(),
        amount,
        random.choice(descriptions)
    ])

    payment_date = posting_date + timedelta(days=random.randint(5, 45))
    payment_rows.append([
        f"PAY{i:05d}",
        source_doc_id,
        invoice_id,
        payment_date.date().isoformat(),
        round(amount * random.uniform(0.95, 1.00), 2)
    ])

invoices = pd.DataFrame(invoice_rows, columns=[
    "invoice_id", "source_doc_id", "vendor_id", "gl_account", "cost_center_id",
    "invoice_date", "posting_date", "amount_usd", "description"
])

payments = pd.DataFrame(payment_rows, columns=[
    "payment_id", "source_doc_id", "invoice_id", "payment_date", "paid_amount_usd"
])

# Introduce a few quality issues
invoices.loc[3, "vendor_id"] = None
invoices.loc[7, "amount_usd"] = -500
invoices.loc[10, "description"] = None

invoices.to_csv(RAW_DIR / "invoices.csv", index=False)
payments.to_csv(RAW_DIR / "payments.csv", index=False)


# Create PySpark job
pyspark_job_code = r'''
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, concat_ws, when, trim, upper
from pathlib import Path

spark = SparkSession.builder.appName("SAPFICOCommonKeyRetrievalPipeline").getOrCreate()

base_path = "data/raw"
out_path = "data/output"

vendors = spark.read.option("header", True).csv(f"{base_path}/vendors.csv")
gl_accounts = spark.read.option("header", True).csv(f"{base_path}/gl_accounts.csv")
cost_centers = spark.read.option("header", True).csv(f"{base_path}/cost_centers.csv")
invoices = spark.read.option("header", True).csv(f"{base_path}/invoices.csv")
payments = spark.read.option("header", True).csv(f"{base_path}/payments.csv")

# Standardization
vendors = vendors.withColumn("vendor_name_clean", upper(trim(col("vendor_name"))))
invoices = invoices.withColumn("description_clean", upper(trim(col("description"))))

# Common key creation
invoices = invoices.withColumn(
    "common_business_key",
    concat_ws("_", col("vendor_id"), col("cost_center_id"), col("gl_account"))
)

payments = payments.withColumn(
    "common_business_key",
    lit(None)
)

# Integrated view
integrated = (
    invoices
    .join(vendors, on="vendor_id", how="left")
    .join(gl_accounts, on="gl_account", how="left")
    .join(cost_centers, on="cost_center_id", how="left")
    .join(payments.select("invoice_id", "payment_id", "payment_date", "paid_amount_usd"), on="invoice_id", how="left")
)

# Data quality checks
dq = integrated.select(
    "invoice_id",
    when(col("vendor_id").isNull(), lit("Missing vendor_id"))
    .when(col("amount_usd").cast("double") <= 0, lit("Invalid amount"))
    .when(col("description").isNull(), lit("Missing description"))
    .otherwise(lit(None)).alias("issue_reason")
).filter(col("issue_reason").isNotNull())

# Retrieval-ready document
retrieval_docs = integrated.withColumn(
    "retrieval_text",
    concat_ws(
        " ",
        lit("Invoice"),
        col("invoice_id"),
        lit("Vendor"),
        col("vendor_name"),
        lit("Cost Center"),
        col("cost_center_name"),
        lit("GL Account"),
        col("gl_account_name"),
        lit("Amount"),
        col("amount_usd"),
        lit("Description"),
        col("description")
    )
)

embedding_input = retrieval_docs.select("invoice_id", "common_business_key", "retrieval_text")


print("Pipeline completed successfully.")
spark.stop()
'''
(PROJECT_DIR / "pyspark_job.py").write_text(pyspark_job_code, encoding="utf-8")


# Create Airflow DAG

dag_code = r'''
from airflow import DAG
from airflow.operators.bash import BashOperator
from datetime import datetime

default_args = {
    "owner": "Ashok",
    "start_date": datetime(2024, 1, 1),
    "retries": 1
}

with DAG(
    dag_id="sap_fico_common_key_retrieval_pipeline",
    default_args=default_args,
    schedule_interval="@daily",
    catchup=False
) as dag:

    start_task = BashOperator(
        task_id="start_pipeline",
        bash_command="echo Starting SAP FICO common key retrieval pipeline"
    )

    run_pyspark_job = BashOperator(
        task_id="run_pyspark_job",
        bash_command="python pyspark_job.py"
    )

    end_task = BashOperator(
        task_id="end_pipeline",
        bash_command="echo Pipeline completed"
    )

    start_task >> run_pyspark_job >> end_task
'''
(DAG_DIR / "sap_fico_common_key_retrieval_dag.py").write_text(dag_code, encoding="utf-8")



# Zip project

zip_path = Path("Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in PROJECT_DIR.rglob("*"):
        if file_path.is_file():
            zf.write(file_path, arcname=file_path.relative_to(PROJECT_DIR))

print("Project created successfully.")
print("\nFiles:")
for p in sorted(PROJECT_DIR.rglob("*")):
    if p.is_file():
        print("-", p)

print("\nZIP created:", zip_path.resolve())

Project created successfully.

Files:
- Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline/dags/sap_fico_common_key_retrieval_dag.py
- Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline/data/raw/cost_centers.csv
- Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline/data/raw/gl_accounts.csv
- Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline/data/raw/invoices.csv
- Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline/data/raw/payments.csv
- Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline/data/raw/vendors.csv
- Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline/pyspark_job.py

ZIP created: /content/Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline.zip


In [ ]:
!pip -q install pyspark

!python Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline/pyspark_job.py

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/30 19:03:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/30 19:03:57 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/raw/vendors.csv.
java.io.FileNotFoundException: File data/raw/vendors.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$

In [ ]:
from pathlib import Path

fixed_pyspark_job_code = r'''
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, concat_ws, when, trim, upper
from pathlib import Path

spark = SparkSession.builder.appName("SAPFICOCommonKeyRetrievalPipeline").getOrCreate()

project_root = "Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline"
base_path = f"{project_root}/data/raw"
out_path = f"{project_root}/data/output"

vendors = spark.read.option("header", True).csv(f"{base_path}/vendors.csv")
gl_accounts = spark.read.option("header", True).csv(f"{base_path}/gl_accounts.csv")
cost_centers = spark.read.option("header", True).csv(f"{base_path}/cost_centers.csv")
invoices = spark.read.option("header", True).csv(f"{base_path}/invoices.csv")
payments = spark.read.option("header", True).csv(f"{base_path}/payments.csv")

vendors = vendors.withColumn("vendor_name_clean", upper(trim(col("vendor_name"))))
invoices = invoices.withColumn("description_clean", upper(trim(col("description"))))

invoices = invoices.withColumn(
    "common_business_key",
    concat_ws("_", col("vendor_id"), col("cost_center_id"), col("gl_account"))
)

integrated = (
    invoices
    .join(vendors, on="vendor_id", how="left")
    .join(gl_accounts, on="gl_account", how="left")
    .join(cost_centers, on="cost_center_id", how="left")
    .join(
        payments.select("invoice_id", "payment_id", "payment_date", "paid_amount_usd"),
        on="invoice_id",
        how="left"
    )
)

dq = integrated.select(
    "invoice_id",
    when(col("vendor_id").isNull(), lit("Missing vendor_id"))
    .when(col("amount_usd").cast("double") <= 0, lit("Invalid amount"))
    .when(col("description").isNull(), lit("Missing description"))
    .otherwise(lit(None)).alias("issue_reason")
).filter(col("issue_reason").isNotNull())

retrieval_docs = integrated.withColumn(
    "retrieval_text",
    concat_ws(
        " ",
        lit("Invoice"),
        col("invoice_id"),
        lit("Vendor"),
        col("vendor_name"),
        lit("Cost Center"),
        col("cost_center_name"),
        lit("GL Account"),
        col("gl_account_name"),
        lit("Amount"),
        col("amount_usd"),
        lit("Description"),
        col("description")
    )
)

embedding_input = retrieval_docs.select("invoice_id", "common_business_key", "retrieval_text")

integrated.toPandas().to_csv(f"{out_path}/integrated_finance_view.csv", index=False)
dq.toPandas().to_csv(f"{out_path}/data_quality_issues.csv", index=False)
retrieval_docs.select("invoice_id", "common_business_key", "retrieval_text").toPandas().to_csv(
    f"{out_path}/retrieval_ready_documents.csv", index=False
)
embedding_input.toPandas().to_csv(f"{out_path}/embedding_input.csv", index=False)

print("Pipeline completed successfully.")
spark.stop()
'''

project_file = Path("Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline/pyspark_job.py")
project_file.write_text(fixed_pyspark_job_code, encoding="utf-8")

print("pyspark_job.py fixed successfully")

pyspark_job.py fixed successfully


In [ ]:
!pip -q install pyspark
!python Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline/pyspark_job.py

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/30 19:04:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Pipeline completed successfully.


In [ ]:
from pathlib import Path

OUT_DIR = Path("Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline/data/output")

print("Files in output folder:")
for p in OUT_DIR.glob("*"):
    print("-", p.name)

Files in output folder:
- embedding_input.csv
- data_quality_issues.csv
- retrieval_ready_documents.csv
- integrated_finance_view.csv


In [ ]:
import pandas as pd
from pathlib import Path

OUT_DIR = Path("Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline/data/output")

for file_name in [
    "integrated_finance_view.csv",
    "data_quality_issues.csv",
    "retrieval_ready_documents.csv",
    "embedding_input.csv"
]:
    print("\n" + "="*80)
    print(file_name)
    df = pd.read_csv(OUT_DIR / file_name)
    display(df.head())


integrated_finance_view.csv


,invoice_id,cost_center_id,gl_account,vendor_id,source_doc_id,invoice_date,posting_date,amount_usd,description,description_clean,...,vendor_category,vendor_state,vendor_name_clean,gl_account_name,gl_type,cost_center_name,department,payment_id,payment_date,paid_amount_usd
0,INV00001,CC110,500300,V002,SRC0001,2024-11-21,2024-11-23,2225.53,field contractor invoice,FIELD CONTRACTOR INVOICE,...,IT Services,CT,BLUEGRID TECHNOLOGIES,Opex - Contractors,Expense,Distribution Maintenance,Operations,PAY00001,2024-12-13,2139.09
1,INV00002,CC140,400100,V001,SRC0002,2024-02-13,2024-02-22,5260.00,substation maintenance work,SUBSTATION MAINTENANCE WORK,...,Maintenance,NY,ALPHA ENERGY SERVICES,Revenue - Energy Delivery,Revenue,Finance Operations,Finance,PAY00002,2024-02-29,5004.84
2,INV00003,CC100,500100,V003,SRC0003,2024-04-17,2024-04-21,25762.41,transformer repair service,TRANSFORMER REPAIR SERVICE,...,Electrical Components,MA,NORTHWIND ELECTRIC,Opex - Maintenance,Expense,Transmission Operations,Operations,PAY00003,2024-05-31,24730.42
3,INV00004,CC110,400100,NaN,SRC0004,2024-11-19,2024-11-28,21556.47,substation maintenance work,SUBSTATION MAINTENANCE WORK,...,NaN,NaN,NaN,Revenue - Energy Delivery,Revenue,Distribution Maintenance,Operations,PAY00004,2025-01-09,20778.49
4,INV00005,CC120,500400,V004,SRC0005,2024-01-01,2024-01-04,35208.83,field contractor invoice,FIELD CONTRACTOR INVOICE,...,Field Services,PA,GREENVOLT CONTRACTORS,Opex - Logistics,Expense,Substation Engineering,Engineering,PAY00005,2024-01-26,33722.10



data_quality_issues.csv


,invoice_id,issue_reason
0,INV00004,Missing vendor_id
1,INV00008,Invalid amount
2,INV00011,Missing description



retrieval_ready_documents.csv


,invoice_id,common_business_key,retrieval_text
0,INV00001,V002_CC110_500300,Invoice INV00001 Vendor BlueGrid Technologies ...
1,INV00002,V001_CC140_400100,Invoice INV00002 Vendor Alpha Energy Services ...
2,INV00003,V003_CC100_500100,Invoice INV00003 Vendor Northwind Electric Cos...
3,INV00004,CC110_400100,Invoice INV00004 Vendor Cost Center Distributi...
4,INV00005,V004_CC120_500400,Invoice INV00005 Vendor GreenVolt Contractors ...



embedding_input.csv


,invoice_id,common_business_key,retrieval_text
0,INV00001,V002_CC110_500300,Invoice INV00001 Vendor BlueGrid Technologies ...
1,INV00002,V001_CC140_400100,Invoice INV00002 Vendor Alpha Energy Services ...
2,INV00003,V003_CC100_500100,Invoice INV00003 Vendor Northwind Electric Cos...
3,INV00004,CC110_400100,Invoice INV00004 Vendor Cost Center Distributi...
4,INV00005,V004_CC120_500400,Invoice INV00005 Vendor GreenVolt Contractors ...


In [ ]:
!pip install -q apache-airflow-providers-databricks==7.7.4 requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 6.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.9/86.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.7/305.7 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.1/92.1 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.9/213.9 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.8/152.8 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.7/59.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.7/74.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━

In [ ]:
import requests
print("Install check passed")

Install check passed


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, concat_ws, when, trim, upper

spark = SparkSession.builder.appName("SAPFICODatabricksStyleTest").getOrCreate()

project_root = "Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline"
base_path = f"{project_root}/data/raw"
out_path = f"{project_root}/data/output"

vendors = spark.read.option("header", True).csv(f"{base_path}/vendors.csv")
gl_accounts = spark.read.option("header", True).csv(f"{base_path}/gl_accounts.csv")
cost_centers = spark.read.option("header", True).csv(f"{base_path}/cost_centers.csv")
invoices = spark.read.option("header", True).csv(f"{base_path}/invoices.csv")
payments = spark.read.option("header", True).csv(f"{base_path}/payments.csv")

vendors = vendors.withColumn("vendor_name_clean", upper(trim(col("vendor_name"))))
invoices = invoices.withColumn("description_clean", upper(trim(col("description"))))

invoices = invoices.withColumn(
    "common_business_key",
    concat_ws("_", col("vendor_id"), col("cost_center_id"), col("gl_account"))
)

integrated = (
    invoices
    .join(vendors, on="vendor_id", how="left")
    .join(gl_accounts, on="gl_account", how="left")
    .join(cost_centers, on="cost_center_id", how="left")
    .join(
        payments.select("invoice_id", "payment_id", "payment_date", "paid_amount_usd"),
        on="invoice_id",
        how="left"
    )
)

dq = integrated.select(
    "invoice_id",
    when(col("vendor_id").isNull(), lit("Missing vendor_id"))
    .when(col("amount_usd").cast("double") <= 0, lit("Invalid amount"))
    .when(col("description").isNull(), lit("Missing description"))
    .otherwise(lit(None)).alias("issue_reason")
).filter(col("issue_reason").isNotNull())

retrieval_docs = integrated.withColumn(
    "retrieval_text",
    concat_ws(
        " ",
        lit("Invoice"),
        col("invoice_id"),
        lit("Vendor"),
        col("vendor_name"),
        lit("Cost Center"),
        col("cost_center_name"),
        lit("GL Account"),
        col("gl_account_name"),
        lit("Amount"),
        col("amount_usd"),
        lit("Description"),
        col("description")
    )
)

embedding_input = retrieval_docs.select("invoice_id", "common_business_key", "retrieval_text")

integrated.show(5, truncate=False)
dq.show(truncate=False)
embedding_input.show(5, truncate=False)

spark.stop()

+----------+--------------+----------+---------+-------------+------------+------------+----------+---------------------------+---------------------------+-------------------+---------------------+---------------------+------------+---------------------+-------------------------+-------+------------------------+-----------+----------+------------+---------------+
|invoice_id|cost_center_id|gl_account|vendor_id|source_doc_id|invoice_date|posting_date|amount_usd|description                |description_clean          |common_business_key|vendor_name          |vendor_category      |vendor_state|vendor_name_clean    |gl_account_name          |gl_type|cost_center_name        |department |payment_id|payment_date|paid_amount_usd|
+----------+--------------+----------+---------+-------------+------------+------------+----------+---------------------------+---------------------------+-------------------+---------------------+---------------------+------------+---------------------+--------------

In [ ]:
from airflow import DAG
from airflow.providers.standard.operators.empty import EmptyOperator
from airflow.providers.databricks.operators.databricks import (
    DatabricksCreateJobsOperator,
    DatabricksRunNowOperator,
)
from datetime import datetime

default_args = {
    "owner": "Ashok",
    "start_date": datetime(2024, 1, 1),
    "retries": 1
}

job_config = {
    "name": "sap_fico_common_key_retrieval_job",
    "tasks": [
        {
            "task_key": "run_sap_fico_notebook",
            "notebook_task": {
                "notebook_path": "/Workspace/Users/your_email@company.com/pyspark_job_databricks"
            },
            "new_cluster": {
                "spark_version": "15.4.x-scala2.12",
                "node_type_id": "Standard_DS3_v2",
                "num_workers": 1
            }
        }
    ]
}

with DAG(
    dag_id="sap_fico_airflow_to_databricks_trigger",
    default_args=default_args,
    schedule="@daily",
    catchup=False,
    tags=["airflow", "databricks", "sap_fico"]
) as dag:

    start_task = EmptyOperator(
        task_id="start_pipeline"
    )

    create_or_update_job = DatabricksCreateJobsOperator(
        task_id="create_or_update_databricks_job",
        databricks_conn_id="databricks_default",
        json=job_config
    )

    trigger_databricks_job = DatabricksRunNowOperator(
        task_id="trigger_databricks_job",
        databricks_conn_id="databricks_default",
        job_id="{{ ti.xcom_pull(task_ids='create_or_update_databricks_job')['job_id'] }}"
    )

    end_task = EmptyOperator(
        task_id="end_pipeline"
    )

    start_task >> create_or_update_job >> trigger_databricks_job >> end_task

2026-03-30T19:10:33.464351Z [info     ] setup plugin alembic.autogenerate.schemas [alembic.runtime.plugins] loc=plugins.py:37
2026-03-30T19:10:33.465579Z [info     ] setup plugin alembic.autogenerate.tables [alembic.runtime.plugins] loc=plugins.py:37
2026-03-30T19:10:33.466679Z [info     ] setup plugin alembic.autogenerate.types [alembic.runtime.plugins] loc=plugins.py:37
2026-03-30T19:10:33.467813Z [info     ] setup plugin alembic.autogenerate.constraints [alembic.runtime.plugins] loc=plugins.py:37
2026-03-30T19:10:33.468648Z [info     ] setup plugin alembic.autogenerate.defaults [alembic.runtime.plugins] loc=plugins.py:37
2026-03-30T19:10:33.469910Z [info     ] setup plugin alembic.autogenerate.comments [alembic.runtime.plugins] loc=plugins.py:37


In [ ]:
from pathlib import Path

DAG_DIR = Path("Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline/dags")
DAG_DIR.mkdir(parents=True, exist_ok=True)

dag_code = r'''
from airflow import DAG
from airflow.providers.standard.operators.empty import EmptyOperator
from airflow.providers.databricks.operators.databricks import (
    DatabricksCreateJobsOperator,
    DatabricksRunNowOperator,
)
from datetime import datetime

default_args = {
    "owner": "ranjith",
    "start_date": datetime(2024, 1, 1),
    "retries": 1
}

job_config = {
    "name": "sap_fico_common_key_retrieval_job",
    "tasks": [
        {
            "task_key": "run_sap_fico_notebook",
            "notebook_task": {
                "notebook_path": "/Workspace/Users/your_email@company.com/pyspark_job_databricks"
            },
            "new_cluster": {
                "spark_version": "15.4.x-scala2.12",
                "node_type_id": "Standard_DS3_v2",
                "num_workers": 1
            }
        }
    ]
}

with DAG(
    dag_id="sap_fico_airflow_to_databricks_trigger",
    default_args=default_args,
    schedule="@daily",
    catchup=False,
    tags=["airflow", "databricks", "sap_fico"]
) as dag:

    start_task = EmptyOperator(
        task_id="start_pipeline"
    )

    create_or_update_job = DatabricksCreateJobsOperator(
        task_id="create_or_update_databricks_job",
        databricks_conn_id="databricks_default",
        json=job_config
    )

    trigger_databricks_job = DatabricksRunNowOperator(
        task_id="trigger_databricks_job",
        databricks_conn_id="databricks_default",
        job_id="{{ ti.xcom_pull(task_ids='create_or_update_databricks_job')['job_id'] }}"
    )

    end_task = EmptyOperator(
        task_id="end_pipeline"
    )

    start_task >> create_or_update_job >> trigger_databricks_job >> end_task
'''

(DAG_DIR / "sap_fico_airflow_to_databricks_trigger.py").write_text(dag_code, encoding="utf-8")
print("DAG file created successfully")

DAG file created successfully


In [ ]:
from pathlib import Path

PROJECT_DIR = Path("Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline")
DAG_DIR = PROJECT_DIR / "dags"
DAG_DIR.mkdir(parents=True, exist_ok=True)

updated_dag_code = r'''
from airflow import DAG
from airflow.providers.standard.operators.empty import EmptyOperator
from airflow.providers.databricks.operators.databricks import (
    DatabricksCreateJobsOperator,
    DatabricksRunNowOperator,
)
from datetime import datetime

default_args = {
    "owner": "Ashok",
    "start_date": datetime(2024, 1, 1),
    "retries": 1
}

job_config = {
    "name": "sap_fico_common_key_retrieval_job",
    "tasks": [
        {
            "task_key": "run_sap_fico_notebook",
            "notebook_task": {
                "notebook_path": "/Workspace/Users/your_email@company.com/pyspark_job_databricks"
            },
            "new_cluster": {
                "spark_version": "15.4.x-scala2.12",
                "node_type_id": "Standard_DS3_v2",
                "num_workers": 1
            }
        }
    ]
}

with DAG(
    dag_id="sap_fico_airflow_to_databricks_trigger",
    default_args=default_args,
    schedule="@daily",
    catchup=False,
    tags=["airflow", "databricks", "sap_fico"]
) as dag:

    start_task = EmptyOperator(task_id="start_pipeline")

    create_or_update_job = DatabricksCreateJobsOperator(
        task_id="create_or_update_databricks_job",
        databricks_conn_id="databricks_default",
        json=job_config
    )

    trigger_databricks_job = DatabricksRunNowOperator(
        task_id="trigger_databricks_job",
        databricks_conn_id="databricks_default",
        job_id="{{ ti.xcom_pull(task_ids='create_or_update_databricks_job')['job_id'] }}"
    )

    end_task = EmptyOperator(task_id="end_pipeline")

    start_task >> create_or_update_job >> trigger_databricks_job >> end_task
'''

dag_file = DAG_DIR / "sap_fico_airflow_to_databricks_trigger.py"
dag_file.write_text(updated_dag_code, encoding="utf-8")

print("Updated DAG written successfully")
print("File path:", dag_file.resolve())

Updated DAG written successfully
File path: /content/Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline/dags/sap_fico_airflow_to_databricks_trigger.py


In [ ]:
from pathlib import Path

dag_file = Path("Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline/dags/sap_fico_airflow_to_databricks_trigger.py")

real_notebook_path = "/Workspace/Users/your_email@company.com/pyspark_job_databricks"

dag_text = dag_file.read_text(encoding="utf-8")
dag_text = dag_text.replace(
    "/Workspace/Users/your_email@company.com/pyspark_job_databricks",
    real_notebook_path
)

dag_file.write_text(dag_text, encoding="utf-8")

print("Notebook path updated successfully")
print("Updated DAG file:", dag_file.resolve())

Notebook path updated successfully
Updated DAG file: /content/Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline/dags/sap_fico_airflow_to_databricks_trigger.py


In [ ]:
from pathlib import Path

dag_file = Path("Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline/dags/sap_fico_airflow_to_databricks_trigger.py")

print("Reading DAG file:\n")
print(dag_file.read_text(encoding="utf-8"))

Reading DAG file:


from airflow import DAG
from airflow.providers.standard.operators.empty import EmptyOperator
from airflow.providers.databricks.operators.databricks import (
    DatabricksCreateJobsOperator,
    DatabricksRunNowOperator,
)
from datetime import datetime

default_args = {
    "owner": "Ashok",
    "start_date": datetime(2024, 1, 1),
    "retries": 1
}

job_config = {
    "name": "sap_fico_common_key_retrieval_job",
    "tasks": [
        {
            "task_key": "run_sap_fico_notebook",
            "notebook_task": {
                "notebook_path": "/Workspace/Users/your_email@company.com/pyspark_job_databricks"
            },
            "new_cluster": {
                "spark_version": "15.4.x-scala2.12",
                "node_type_id": "Standard_DS3_v2",
                "num_workers": 1
            }
        }
    ]
}

with DAG(
    dag_id="sap_fico_airflow_to_databricks_trigger",
    default_args=default_args,
    schedule="@daily",
    catchup=False,
    tags=[

In [ ]:
import zipfile
from pathlib import Path

PROJECT_DIR = Path("Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline")
zip_path = Path("Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline-updated.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in PROJECT_DIR.rglob("*"):
        if file_path.is_file():
            zf.write(file_path, arcname=file_path.relative_to(PROJECT_DIR))

print("Updated project zipped successfully")
print("ZIP path:", zip_path.resolve())

Updated project zipped successfully
ZIP path: /content/Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline-updated.zip


In [ ]:
from pathlib import Path

PROJECT_DIR = Path("Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline")

print("Project files:")
for p in sorted(PROJECT_DIR.rglob("*")):
    if p.is_file():
        print("-", p)

Project files:
- Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline/dags/sap_fico_airflow_to_databricks_trigger.py
- Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline/data/raw/cost_centers.csv
- Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline/data/raw/gl_accounts.csv
- Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline/data/raw/invoices.csv
- Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline/data/raw/payments.csv
- Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline/data/raw/vendors.csv
- Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline/pyspark_job.py


In [ ]:
from pathlib import Path

DAG_DIR = Path("Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline/dags")

print("DAG files:")
for p in DAG_DIR.glob("*.py"):
    print("-", p.name)

DAG files:
- sap_fico_airflow_to_databricks_trigger.py


In [ ]:
from airflow import DAG
from airflow.providers.standard.operators.empty import EmptyOperator
from airflow.providers.databricks.operators.databricks import (
    DatabricksCreateJobsOperator,
    DatabricksRunNowOperator,
)
from datetime import datetime

default_args = {
    "owner": "ranjith",
    "start_date": datetime(2024, 1, 1),
    "retries": 1
}

job_config = {
    "name": "sap_fico_common_key_retrieval_job",
    "tasks": [
        {
            "task_key": "run_sap_fico_notebook",
            "notebook_task": {
                "notebook_path": "/Workspace/Users/your_email@company.com/pyspark_job_databricks"
            },
            "new_cluster": {
                "spark_version": "15.4.x-scala2.12",
                "node_type_id": "Standard_DS3_v2",
                "num_workers": 1
            }
        }
    ]
}

with DAG(
    dag_id="sap_fico_airflow_to_databricks_trigger",
    default_args=default_args,
    schedule="@daily",
    catchup=False,
    tags=["airflow", "databricks", "sap_fico"]
) as dag:

    start_task = EmptyOperator(task_id="start_pipeline")

    create_or_update_job = DatabricksCreateJobsOperator(
        task_id="create_or_update_databricks_job",
        databricks_conn_id="databricks_default",
        json=job_config
    )

    trigger_databricks_job = DatabricksRunNowOperator(
        task_id="trigger_databricks_job",
        databricks_conn_id="databricks_default",
        job_id="{{ ti.xcom_pull(task_ids='create_or_update_databricks_job')['job_id'] }}"
    )

    end_task = EmptyOperator(task_id="end_pipeline")

    start_task >> create_or_update_job >> trigger_databricks_job >> end_task

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline")
DAG_DIR = PROJECT_DIR / "dags"
DAG_DIR.mkdir(parents=True, exist_ok=True)

final_dag_code = r'''
from airflow import DAG
from airflow.providers.standard.operators.empty import EmptyOperator
from airflow.providers.databricks.operators.databricks import (
    DatabricksCreateJobsOperator,
    DatabricksRunNowOperator,
)
from datetime import datetime

default_args = {
    "owner": "Ashok",
    "start_date": datetime(2024, 1, 1),
    "retries": 1
}

job_config = {
    "name": "sap_fico_common_key_retrieval_job",
    "tasks": [
        {
            "task_key": "run_sap_fico_notebook",
            "notebook_task": {
                "notebook_path": "/Workspace/Users/your_email@company.com/pyspark_job_databricks"
            },
            "new_cluster": {
                "spark_version": "15.4.x-scala2.12",
                "node_type_id": "Standard_DS3_v2",
                "num_workers": 1
            }
        }
    ]
}

with DAG(
    dag_id="sap_fico_airflow_to_databricks_trigger",
    default_args=default_args,
    schedule="@daily",
    catchup=False,
    tags=["airflow", "databricks", "sap_fico"]
) as dag:

    start_task = EmptyOperator(task_id="start_pipeline")

    create_or_update_job = DatabricksCreateJobsOperator(
        task_id="create_or_update_databricks_job",
        databricks_conn_id="databricks_default",
        json=job_config
    )

    trigger_databricks_job = DatabricksRunNowOperator(
        task_id="trigger_databricks_job",
        databricks_conn_id="databricks_default",
        job_id="{{ ti.xcom_pull(task_ids='create_or_update_databricks_job')['job_id'] }}"
    )

    end_task = EmptyOperator(task_id="end_pipeline")

    start_task >> create_or_update_job >> trigger_databricks_job >> end_task
'''

dag_file = DAG_DIR / "sap_fico_airflow_to_databricks_trigger.py"
dag_file.write_text(final_dag_code, encoding="utf-8")

print("Final DAG saved successfully")
print(dag_file.resolve())

Final DAG saved successfully
/content/Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline/dags/sap_fico_airflow_to_databricks_trigger.py


In [ ]:
from pathlib import Path

dag_file = Path("Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline/dags/sap_fico_airflow_to_databricks_trigger.py")

print(dag_file.read_text(encoding="utf-8"))


from airflow import DAG
from airflow.providers.standard.operators.empty import EmptyOperator
from airflow.providers.databricks.operators.databricks import (
    DatabricksCreateJobsOperator,
    DatabricksRunNowOperator,
)
from datetime import datetime

default_args = {
    "owner": "Ashok",
    "start_date": datetime(2024, 1, 1),
    "retries": 1
}

job_config = {
    "name": "sap_fico_common_key_retrieval_job",
    "tasks": [
        {
            "task_key": "run_sap_fico_notebook",
            "notebook_task": {
                "notebook_path": "/Workspace/Users/your_email@company.com/pyspark_job_databricks"
            },
            "new_cluster": {
                "spark_version": "15.4.x-scala2.12",
                "node_type_id": "Standard_DS3_v2",
                "num_workers": 1
            }
        }
    ]
}

with DAG(
    dag_id="sap_fico_airflow_to_databricks_trigger",
    default_args=default_args,
    schedule="@daily",
    catchup=False,
    tags=["airflow", "databri

In [ ]:
import zipfile
from pathlib import Path

PROJECT_DIR = Path("Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline")
zip_path = Path("Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline-final.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in PROJECT_DIR.rglob("*"):
        if file_path.is_file():
            zf.write(file_path, arcname=file_path.relative_to(PROJECT_DIR))

print("Final project zipped successfully")
print("ZIP path:", zip_path.resolve())

Final project zipped successfully
ZIP path: /content/Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline-final.zip


In [ ]:
from pathlib import Path

zip_path = Path("Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline-final.zip")

if zip_path.exists():
    print("ZIP file created:", zip_path.resolve())
else:
    print("ZIP file not found")

ZIP file created: /content/Airflow-PySpark-SAP-FICO-Common-Key-Retrieval-Pipeline-final.zip
